In [1]:
import pandas as pd

In [16]:
df_subqueries = pd.read_table("/home/user/mayer/FactorJoin/End-to-End-CardEst-Benchmark/workloads/stats_CEB/sub_plan_queries/stats_CEB_sub_queries.sql", sep='\|\|', header=None)
df_subqueries.columns = ['query', 'query_id']
df_subqueries.reset_index(drop=False, inplace=True)
df_subqueries["query_id"] = df_subqueries["query_id"] + 1

df_estimates = pd.read_table("/home/user/mayer/FactorJoin/checkpoints/stats_CEB_sub_queries_model_stats_greedy_200.txt", header=None)
df_estimates.reset_index(drop=False, inplace=True)
df_estimates.rename(columns={0: 'estimate'}, inplace=True)

df_subqueries = df_subqueries.merge(df_estimates, left_on='index', right_on="index")
df_subqueries

/home/user/mayer/miniconda3/envs/factorjoin/lib/python3.7/site-packages/pandas/util/_decorators.py:311: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  return func(*args, **kwargs)


,index,query,query_id,estimate
0,0,"SELECT COUNT(*) FROM users as u, badges as b W...",1,7.985100e+04
1,1,"SELECT COUNT(*) FROM badges as b, comments as ...",2,1.229649e+07
2,2,"SELECT COUNT(*) FROM postHistory as ph, commen...",3,4.125085e+06
3,3,"SELECT COUNT(*) FROM postHistory as ph, commen...",4,4.240229e+06
4,4,"SELECT COUNT(*) FROM votes as v, comments as c...",5,1.071971e+07
...,...,...,...,...
2598,2598,"SELECT COUNT(*) FROM users as u, comments as c...",146,1.289724e+06
2599,2599,"SELECT COUNT(*) FROM users as u, comments as c...",146,1.308301e+06
2600,2600,"SELECT COUNT(*) FROM users as u, comments as c...",146,6.994912e+06
2601,2601,"SELECT COUNT(*) FROM users as u, posts as p, p...",146,7.983313e+05


In [21]:
def extract_tables(from_statement):
    tables = []
    for table in from_statement.split(","):
        table = table.strip()
        table = table.split("as")[1].strip()
        
        tables.append(table)
        
    tables = " ".join(tables).upper()
    return tables

In [23]:
df_subqueries["tables"] = df_subqueries["query"].str.extract(r'FROM (.*) WHERE')
df_subqueries["tables"] = df_subqueries["tables"].apply(extract_tables)
# drop index column
try:
    df_subqueries.drop(columns=['index'], inplace=True)
except:
    pass
df_subqueries.to_csv("stats_subqueries.csv", index=False)
df_subqueries

,query,query_id,estimate,tables
0,"SELECT COUNT(*) FROM users as u, badges as b W...",1,7.985100e+04,U B
1,"SELECT COUNT(*) FROM badges as b, comments as ...",2,1.229649e+07,B C U
2,"SELECT COUNT(*) FROM postHistory as ph, commen...",3,4.125085e+06,PH C U
3,"SELECT COUNT(*) FROM postHistory as ph, commen...",4,4.240229e+06,PH C U
4,"SELECT COUNT(*) FROM votes as v, comments as c...",5,1.071971e+07,V C U
...,...,...,...,...
2598,"SELECT COUNT(*) FROM users as u, comments as c...",146,1.289724e+06,U C P PL PH
2599,"SELECT COUNT(*) FROM users as u, comments as c...",146,1.308301e+06,U C P PL V
2600,"SELECT COUNT(*) FROM users as u, comments as c...",146,6.994912e+06,U C P PH V
2601,"SELECT COUNT(*) FROM users as u, posts as p, p...",146,7.983313e+05,U P PL PH V
